# **Experiment Notebook**



---
## Setup Environment

In [1]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd

from utstd.folders import *
from utstd.ipyrenders import *

at = AtFolder(
    course_code=36106,
    assignment="AT1",
)
at.run()

import warnings
warnings.simplefilter(action='ignore')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 48.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.11 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
hdbscan 0.8.41 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
Mounted at /content/gdrive

You can now save your data files in: /content/gdrive/MyDrive/36106/assignment/AT1/data


---
## Student Information

In [2]:
student_name = "SUSHRUTA GANGADHAR PATIL"
student_id = "26273312"

In [3]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_name', value=student_name)

In [4]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

### 0.b Import Packages

In [5]:
# DO NOT MODIFY THE CODE IN THIS CELL
import pandas as pd
import altair as alt

In [6]:
import numpy as np
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, loguniform
from sklearn.metrics import root_mean_squared_error as rmse

---
## A. Experiment Description

In [7]:
# DO NOT MODIFY THE CODE IN THIS CELL
experiment_id = "2"
print_tile(size="h1", key='experiment_id', value=experiment_id)

In [8]:
experiment_hypothesis = """

In Experiment 1 Linear Regression achieved a validation RMSE of $27,162.40. However as Linear
Regression has no regularisations, it fits the training data as closely as possible without any
constraints on the coefficients. For example coefficient of brand_bodytype_encoded in linear
regression model was 13,040. ElasticNet would penalise a coefficient that large and shrink it,
reducing the risk of overfitting to that one feature.

The hypothesis is that by tuning alpha and l1_ratio we can find a model that generalises better to
unseen data and achieves an overall lower validation RMSE than Experiment 1.

"""

In [9]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='experiment_hypothesis', value=experiment_hypothesis)

In [10]:
experiment_expectations = """

I expect ElasticNet configurations to outperform Linear Regression from Experiment 1. Models with
very high alpha will underfit and perform bad, models with very low alpha will behave
similar to regular linear regression. So, we should identify the right value of aplha and l1_ratio
for the model to perform its best.

"""

In [11]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='experiment_expectations', value=experiment_expectations)

---
## B. Feature Selection


In [12]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Load data
try:
  X_train = pd.read_csv(at.folder_path / 'X_train.csv')
  y_train = pd.read_csv(at.folder_path / 'y_train.csv')

  X_val = pd.read_csv(at.folder_path / 'X_val.csv')
  y_val = pd.read_csv(at.folder_path / 'y_val.csv')

  X_test = pd.read_csv(at.folder_path / 'X_test.csv')
  y_test = pd.read_csv(at.folder_path / 'y_test.csv')
except Exception as e:
  print(e)

In [13]:
feature_selection_explanations = """

For this experiment I am using the same 6 features prepared in the
Preparation notebook:

1. kilometres_driven        - how far the car has been driven
2. manufacturing_year       - the year the car was manufactured
3. engine_cylinders         - number of engine cylinders
4. brand_bodytype_encoded   - smoothed mean encoded combination
                              of brand and body type
5. body_drive_encoded       - smoothed mean encoded combination
                              of body type and drive type
6. transmission_type_Manual - binary flag, 1 = Manual, 0 = Automatic

This is to keep the comparision between the models same. The only difference should be the type of
model itself, so if any change in RMSE, it comes entirely from ElasticNet and its regularisation,
not from different features.

"""

In [14]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_explanations', value=feature_selection_explanations)

---
## C. Train Machine Learning Model

### C.1 Import Algorithm



In [15]:
from sklearn.linear_model import ElasticNet

In [16]:
algorithm_selection_explanations = """

For the 2nd experiment, we are selecting ElasticNet. It is a regularised linear regression model
that combines L1 (Lasso) and L2 (Ridge) penalties. Unlike regular Linear Regression which tries to
fit the training data as closely as possible, ElasticNet adds a penalty for large coefficients which
can help the model to generalise better to unseen data.

It is a good next step to Linear Regression as now we can control over how aggressively the model
fits the training data. We already proved the linear relationship of the features to the target
variable, so with ElasticNet we will be able to penalise higher coefficient values and try on
improving the RMSE.

"""

In [17]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='algorithm_selection_explanations', value=algorithm_selection_explanations)

### C.2 Set Hyperparameters


In [18]:
# alpha on log scale from 0.001 to 100 to cover light to heavy regularisation
# l1_ratio from 0 to 1, covers full range from pure L2 to pure L1

param_dist = {
    "alpha": loguniform(0.001, 100),
    "l1_ratio": uniform(0, 1)
}

In [19]:
hyperparameters_selection_explanations = """

There are two hyperparameters that are being tuned for ElasticNet - alpha and l1_ratio.
Alpha is set to loguniform(0.001,100) as the optimal value could be anywhere across a very wide
range of values and log scale ensures that every value in the range is tested throughly without
ignoring the smaller numbers.
l1_ratio is set to uniform(0,1) as the values lie from 0 being pure ridge and 1 being pure lasso and
anything in between being the mix of both behaviours. Log scale is not used here as (0,1) is a valid
range by itself and every value will be treated equakky and tested.

"""

In [20]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='hyperparameters_selection_explanations', value=hyperparameters_selection_explanations)

### C.3 Fit Model

In [21]:
results_iter = []

for n_iter in range(50, 201, 10):
    rs = RandomizedSearchCV(
        ElasticNet(max_iter=10000),
        param_distributions=param_dist,
        n_iter=n_iter,
        scoring="neg_root_mean_squared_error",
        cv=5,
        random_state=42,
        n_jobs=-1
    )
    rs.fit(X_train, y_train)
    results_iter.append({
        "n_iter": n_iter,
        "cv_rmse": round(-rs.best_score_, 2),
        "alpha": round(rs.best_params_["alpha"], 4),
        "l1_ratio": round(rs.best_params_["l1_ratio"], 4)
    })
    print(f"n_iter={n_iter} — alpha={rs.best_params_['alpha']:.4f} | l1_ratio={rs.best_params_['l1_ratio']:.4f} | CV RMSE: ${-rs.best_score_:,.2f}")

n_iter=50 — alpha=0.4108 | l1_ratio=0.4275 | CV RMSE: $19,007.06
n_iter=60 — alpha=1.4690 | l1_ratio=0.8715 | CV RMSE: $18,994.83
n_iter=70 — alpha=1.4690 | l1_ratio=0.8715 | CV RMSE: $18,994.83
n_iter=80 — alpha=1.4690 | l1_ratio=0.8715 | CV RMSE: $18,994.83
n_iter=90 — alpha=1.4690 | l1_ratio=0.8715 | CV RMSE: $18,994.83
n_iter=100 — alpha=1.4690 | l1_ratio=0.8715 | CV RMSE: $18,994.83
n_iter=110 — alpha=0.5542 | l1_ratio=0.6919 | CV RMSE: $18,993.18
n_iter=120 — alpha=0.5542 | l1_ratio=0.6919 | CV RMSE: $18,993.18
n_iter=130 — alpha=0.5542 | l1_ratio=0.6919 | CV RMSE: $18,993.18
n_iter=140 — alpha=0.5542 | l1_ratio=0.6919 | CV RMSE: $18,993.18
n_iter=150 — alpha=0.5542 | l1_ratio=0.6919 | CV RMSE: $18,993.18
n_iter=160 — alpha=0.5542 | l1_ratio=0.6919 | CV RMSE: $18,993.18
n_iter=170 — alpha=0.5542 | l1_ratio=0.6919 | CV RMSE: $18,993.18
n_iter=180 — alpha=0.5542 | l1_ratio=0.6919 | CV RMSE: $18,993.18
n_iter=190 — alpha=0.5542 | l1_ratio=0.6919 | CV RMSE: $18,993.18
n_iter=200 — al

In [22]:
iter_df = pd.DataFrame(results_iter)

alt.Chart(iter_df).mark_line(point=True).encode(
    alt.X("n_iter:Q", title="Number of Iterations"),
    alt.Y("cv_rmse:Q", title="CV RMSE ($)", scale=alt.Scale(zero=False)),
    tooltip=["n_iter", "cv_rmse", "alpha", "l1_ratio"]
).properties(
    title="CV RMSE vs Number of Iterations",
    width=500,
    height=300
).interactive()

alt.Chart(...)

---
## D. Model Evaluation

### D.1 Model Technical Performance

In [23]:
#running the model on best alpha and l1_ratio values obtained
best_model = ElasticNet(
    alpha=0.5542,
    l1_ratio=0.6919,
    max_iter=10000
)
best_model.fit(X_train, y_train)

y_train_pred = best_model.predict(X_train)
y_val_pred = best_model.predict(X_val)

train_rmse = rmse(y_train, y_train_pred)
val_rmse = rmse(y_val, y_val_pred)

print(f"Training RMSE  : ${train_rmse:,.2f}")
print(f"Validation RMSE: ${val_rmse:,.2f}")

Training RMSE  : $20,204.74
Validation RMSE: $27,872.71


In [24]:
results_df = pd.DataFrame([
    {"model": "Baseline", "train_rmse": 26155.62, "val_rmse": 37291.73},
    {"model": "Linear Regression", "train_rmse": 20071.20, "val_rmse": 27162.40},
    {"model": "ElasticNet", "train_rmse": round(train_rmse, 2), "val_rmse": round(val_rmse, 2)}
])

order = ["Baseline", "Linear Regression", "ElasticNet"]

alt.Chart(results_df.melt("model", var_name="split", value_name="rmse")).mark_bar().encode(
    alt.X("model:N", title="", sort=order, axis=alt.Axis(labelAngle=-15)),
    alt.Y("rmse:Q", title="RMSE ($)"),
    alt.Color("split:N", legend=alt.Legend(title="Split")),
    alt.Column("split:N", sort=order, title=""),
    tooltip=["model", "split", "rmse"]
).properties(
    title=alt.TitleParams("ElasticNet vs Baseline vs Linear Regression", anchor="middle"),
    width=200,
    height=300
).interactive()

alt.Chart(...)

In [25]:
coef_df = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": best_model.coef_
}).sort_values("coefficient", ascending=False)

print(f"Intercept: $ {best_model.intercept_[0]:.2f}")
print()
print(coef_df.to_string(index=False))

Intercept: $ 24556.96

                 feature  coefficient
  brand_bodytype_encoded 10734.813950
        engine_cylinders  3366.370843
      body_drive_encoded  2069.729352
transmission_type_Manual  1003.950840
      manufacturing_year   855.494498
       kilometres_driven -5733.646802


In [26]:
# Consistency check across price ranges
y_val_pred_lr = best_model.predict(X_val).flatten()
y_val_actual = y_val.values.flatten()

errors = abs(y_val_actual - y_val_pred_lr)

val_df_lr = pd.DataFrame({
    "actual": y_val_actual,
    "predicted": y_val_pred_lr,
    "error": errors
})

val_df_lr["price_bucket"] = pd.cut(val_df_lr["actual"],
    bins=[0, 15000, 30000, 50000, 100000, float("inf")],
    labels=["<$15k", "$15k-30k", "$30k-50k", "$50k-100k", ">$100k"]
)

print("ElasticNet - RMSE by price range:")
print(val_df_lr.groupby("price_bucket")["error"].agg(
    count="count",
    rmse=lambda x: np.sqrt((x**2).mean())
).round(2).to_string())

ElasticNet - RMSE by price range:
              count       rmse
price_bucket                  
<$15k             4   10788.17
$15k-30k        892    5571.86
$30k-50k       1131   11402.77
$50k-100k       499   26881.47
>$100k           82  134813.55


In [27]:
model_performance_explanations = """

Firstly, I used RandomizedSearchCV to find the optimal alpha and l1_ratio values. It was from 50-200
iterations and cross-validation was done within the training data to check on what values the prices
starts to converge and the CV value stabilises. At n= 110 iteration, the CV converged and stayed flat
till 200 and seeing by the converging trend for other values, it wouldn't converge much further till
larger n values so, I stuck with n=110 being the optimal no of iterations.

Best hyperparameters:
alpha    : 0.5542
l1_ratio : 0.6919
CV RMSE  : $18,993.18

The model was then fit with the best hyperparameters and then set to predict and got the below
results:
Training RMSE  : $20,204.74
Validation RMSE: $27,872.71

Comparing with the other models;
Baseline          — Train: $26,155.62 | Val: $37,291.73
Linear Regression — Train: $20,071.20 | Val: $27,162.40 <- best performer overall
ElasticNet        — Train: $20,204.74 | Val: $27,872.71

ElasticNet performed slightly worse than Linear Regression by $710.31 on validation. This just says
that the features were already well-engineered and did not need much of regularisations.

The effect of regularisation can be seen in the coefficients:
Feature                  | Linear Regression  | ElasticNet  | Change
brand_bodytype_encoded   |             13,040 |      10,734 | shrunk by 2,306
engine_cylinders         |              3,557 |       3,366 | shrunk by 191
transmission_type_Manual |              2,761 |       1,004 | shrunk by 1,757
body_drive_encoded       |              1,091 |       2,069 | increased by 978
manufacturing_year       |                615 |         855 | increased by 240
kilometres_driven        |             -6,879 |      -5,734 | shrunk by 1,145

The large coefficients got shrunk and the weight was redistributed to the  weaker features. No feature
was zeroed out completely which is expected given l1_ratio = 0.6919.

Consistency Analysis compared to Linear Regression:
Price Range  | Linear Reg   | ElasticNet
<$15k        | $11,544.82   | $10,788.17  <- ElasticNet better
$15k-$30k    | $6,278.85    | $5,571.86   <- ElasticNet better
$30k-$50k    | $11,633.54   | $11,402.77  <- ElasticNet better
$50k-$100k   | $25,639.13   | $26,881.47  <- Linear Regression better
>$100k       | $130,247.06  | $134,813.55 <- Linear Regression better

When split into different price brackets, ElasticNet is seen to perform better till $50k range
despite slightly higher RMSE. The overall RMSE is being pulled up by worse performance on expensive
cars.

"""

In [28]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='model_performance_explanations', value=model_performance_explanations)

### D.2 Business Impact from Current Model Performance


In [29]:
business_impacts_explanations = """
For the cars in price range $15k-$30k, ElasticNet outperforms Linear Regression with RMSE of
$5,571.86 vs $6,278.85. For $30k-$50k it is also slightly better at $11,402.77 vs $11,633.54.
The overall RMSE is higher because ElasticNet performs worse on expensive cars above $50k.

Both models struggle above $50k with RMSE exceeding $25,000. This is due to the unavailability of
data on that price range to train on.

So, if I were to choose between these two models for the business use case, i would go with linear
regression as the performance is better than ElasticNet and managable difference in the lower ranges.
But the model as is, cannot be put live or even if put should be under the condition that it should
be fed with more of recent models and expensive cars data as these are the two major factors pulling
the model performance down.
"""

In [30]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='business_impacts_explanations', value=business_impacts_explanations)

## E. Conclusion

In [31]:
experiment_outcome = "Hypothesis Partially Confirmed"

In [32]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h2", key='experiment_outcomes_explanations', value=experiment_outcome)

In [33]:
experiment_results_explanations = """

The hypothesis was that by tuning alpha and l1_ratio we can find a model that generalises better to
unseen data and achieves an overall lower validation RMSE than Experiment 1.

The hypothesis was partially confirmed as ElasticNet did not completely outperform Linear Regression
but it did tend to generalise data better for the prices under $50k and achieved noticeable reduction
in RMSE in comparison to Linear Regression.

Key findings:
RandomizedSearchCV with n_iter=110 has the best hyperparameters and observed no further improvement
beyond that.
Best hyperparameters: alpha=0.5542 and l1_ratio=0.6919 shows the model favours L1 regularisation,
pushing some feature coefficients towards zero.
Regularisation did not help overall but did tend to work better on cars priced under $50k.

Recommendations for Experiment 3:
Try KNN Regression which takes a completely different non-linear approach. Unlike linear models, KNN
does not assume a linear relationship between features and price. This approach may help pick on
patterns that linear models could not.
Test on both p = 1 and 2 to check on which distance measure, manhattan or euclidean distance works
better on this dataset.
choose n value preferably above 2 or 3 for a good generalised model as smaller values like n = 1
would mean choosing itself and just few near ones which could end of overfitting the model and end
up not perform well on the unseen data.

"""

In [34]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h2", key='experiment_results_explanations', value=experiment_results_explanations)

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=fbbefce8-41ae-47c6-bc64-96decd566c0b' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>